<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/03_CNN_LSTM_CHL_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

CNN_DIR = (
    BASE_DIR /
    "06_models" /
    "03_CNN_LSTM"
)

EVALUATION_DIR = CNN_DIR / "evaluation"

PREDICTION_DIR = CNN_DIR / "predictions"

CNN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PREDICTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("STEP 31.9P-14 — CNN-LSTM PROJECT SETUP")
print("=" * 70)

print("\nCNN-LSTM directory:")
print(CNN_DIR)

print("\nEvaluation directory:")
print(EVALUATION_DIR)

print("\nPrediction directory:")
print(PREDICTION_DIR)

print("\n✓ CNN-LSTM directories created.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STEP 31.9P-14 — CNN-LSTM PROJECT SETUP

CNN-LSTM directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM

Evaluation directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/evaluation

Prediction directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/predictions

✓ CNN-LSTM directories created.


In [ ]:
# ======================================================================
# STEP 31.9P-14A — CNN-LSTM GPU VERIFICATION
# ======================================================================

import tensorflow as tf
import sys
import platform

print("=" * 70)
print("STEP 31.9P-14A — CNN-LSTM GPU VERIFICATION")
print("=" * 70)

print("\nPython version:")
print(sys.version)

print("\nTensorFlow version:")
print(tf.__version__)

print("\nPlatform:")
print(platform.platform())

print("\nGPU devices:")
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for gpu in gpus:
        print(gpu)

    print("\n✓ GPU detected.")
    print("✓ CNN-LSTM can use GPU acceleration.")

else:
    print("No GPU detected.")
    print("⚠️ Do NOT start training yet.")

STEP 31.9P-14A — CNN-LSTM GPU VERIFICATION

Python version:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

TensorFlow version:
2.21.0

Platform:
Linux-6.6.122+-x86_64-with-glibc2.39

GPU devices:
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

✓ GPU detected.
✓ CNN-LSTM can use GPU acceleration.


In [ ]:
# ======================================================================
# STEP 31.9P-14A — CNN-LSTM ENVIRONMENT + GPU VERIFICATION
# ======================================================================

import sys
import platform
import os

print("=" * 70)
print("STEP 31.9P-14A — CNN-LSTM ENVIRONMENT + GPU VERIFICATION")
print("=" * 70)

# ----------------------------------------------------------------------
# PYTHON / PLATFORM
# ----------------------------------------------------------------------

print("\nPython version:")
print(sys.version)

print("\nPlatform:")
print(platform.platform())

print("\nCUDA_VISIBLE_DEVICES:")
print(os.environ.get("CUDA_VISIBLE_DEVICES"))

# ----------------------------------------------------------------------
# TENSORFLOW VERIFICATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TENSORFLOW VERIFICATION")
print("=" * 70)

try:
    import tensorflow as tf

    print("TensorFlow version:", tf.__version__)

    gpus = tf.config.list_physical_devices("GPU")

    print("Number of GPUs detected:", len(gpus))

    if gpus:
        print("\nGPU devices:")
        for gpu in gpus:
            print(gpu)

        print("\n✓ TensorFlow GPU detected.")
    else:
        print("\n⚠ No GPU detected by TensorFlow.")

except Exception as e:
    print("\nTensorFlow GPU verification failed:")
    print(e)

# ----------------------------------------------------------------------
# CUDA BUILD INFORMATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TENSORFLOW BUILD INFORMATION")
print("=" * 70)

try:
    print("Built with CUDA:",
          tf.test.is_built_with_cuda())

    print("GPU available:",
          len(tf.config.list_physical_devices("GPU")) > 0)

except Exception as e:
    print("Could not retrieve CUDA build information:")
    print(e)

# ----------------------------------------------------------------------
# SIMPLE GPU COMPUTATION TEST
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("GPU COMPUTATION TEST")
print("=" * 70)

try:
    if gpus:
        with tf.device("/GPU:0"):
            test_a = tf.random.normal((2000, 2000))
            test_b = tf.random.normal((2000, 2000))
            test_c = tf.matmul(test_a, test_b)

        print("GPU matrix multiplication completed.")
        print("Result shape:", test_c.shape)

        # Force computation completion
        _ = test_c.numpy()

        print("✓ GPU computation test successful.")

    else:
        print("GPU computation test skipped because no GPU was detected.")

except Exception as e:
    print("GPU computation test failed:")
    print(e)

print("\n" + "=" * 70)
print("STEP 31.9P-14A COMPLETE")
print("=" * 70)

STEP 31.9P-14A — CNN-LSTM ENVIRONMENT + GPU VERIFICATION

Python version:
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Platform:
Linux-6.6.122+-x86_64-with-glibc2.39

CUDA_VISIBLE_DEVICES:
None

TENSORFLOW VERIFICATION
TensorFlow version: 2.21.0
Number of GPUs detected: 1

GPU devices:
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

✓ TensorFlow GPU detected.

TENSORFLOW BUILD INFORMATION
Built with CUDA: True
GPU available: True

GPU COMPUTATION TEST
GPU matrix multiplication completed.
Result shape: (2000, 2000)
✓ GPU computation test successful.

STEP 31.9P-14A COMPLETE


In [ ]:
# ======================================================================
# STEP 31.9P-14B — CNN-LSTM DATA LOADING & VERIFICATION
# ======================================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 31.9P-14B — CNN-LSTM DATA LOADING & VERIFICATION")
print("=" * 70)

# ----------------------------------------------------------------------
# PROJECT DIRECTORIES
# ----------------------------------------------------------------------

from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREPROCESS_DIR = (
    BASE_DIR /
    "05_model_preprocessing"
)

MODEL_DIR = (
    BASE_DIR /
    "06_models" /
    "03_CNN_LSTM"
)

EVALUATION_DIR = MODEL_DIR / "evaluation"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nProject directory:")
print(BASE_DIR)

print("\nPreprocessing directory:")
print(PREPROCESS_DIR)

print("\nCNN-LSTM model directory:")
print(MODEL_DIR)

print("\nEvaluation directory:")
print(EVALUATION_DIR)

# ----------------------------------------------------------------------
# FILE PATHS
# ----------------------------------------------------------------------

TRAIN_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_train.csv"
)

VALIDATION_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_validation.csv"
)

TEST_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_test.csv"
)

FEATURE_FILE = (
    PREPROCESS_DIR /
    "final_model_feature_columns.csv"
)

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Training dataset": TRAIN_FILE,
    "Validation dataset": VALIDATION_FILE,
    "Test dataset": TEST_FILE,
    "Feature list": FEATURE_FILE
}

for name, path in required_files.items():
    print(f"\n{name}:")
    print(path)
    print("Exists:", path.exists())

    assert path.exists(), f"Missing required file: {path}"

print("\n✓ All required files found.")

# ----------------------------------------------------------------------
# LOAD DATA
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING MODEL-READY DATA")
print("=" * 70)

print("\nLoading training data...")
train_df = pd.read_csv(TRAIN_FILE)

print("Loading validation data...")
validation_df = pd.read_csv(VALIDATION_FILE)

print("Loading test data...")
test_df = pd.read_csv(TEST_FILE)

print("Loading final feature list...")
feature_df = pd.read_csv(FEATURE_FILE)

print("\n" + "=" * 70)
print("DATASET SHAPES")
print("=" * 70)

print("\nTraining:")
print(train_df.shape)

print("\nValidation:")
print(validation_df.shape)

print("\nTest:")
print(test_df.shape)

# ----------------------------------------------------------------------
# FINAL FEATURE LIST
# ----------------------------------------------------------------------

FEATURES = (
    feature_df["feature"]
    .dropna()
    .astype(str)
    .tolist()
)

FEATURES = list(dict.fromkeys(FEATURES))

print("\n" + "=" * 70)
print("FINAL FEATURE CONFIGURATION")
print("=" * 70)

print("\nNumber of selected features:")
print(len(FEATURES))

assert len(FEATURES) == 69

print("\n✓ Exactly 69 final predictors loaded.")

# ----------------------------------------------------------------------
# TARGET CONFIGURATION
# ----------------------------------------------------------------------

TARGET_LOG = "chlorophyll_a_log1p"
TARGET_ORIGINAL = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET CONFIGURATION")
print("=" * 70)

print("\nLog target:")
print(TARGET_LOG)

print("\nOriginal target:")
print(TARGET_ORIGINAL)

# ----------------------------------------------------------------------
# FEATURE EXISTENCE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE EXISTENCE CHECK")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    missing_features = [
        feature for feature in FEATURES
        if feature not in df.columns
    ]

    print(f"\n{dataset_name} missing features:")
    print(len(missing_features))

    assert len(missing_features) == 0

    print(f"✓ All 69 features available in {dataset_name}.")

# ----------------------------------------------------------------------
# TARGET EXISTENCE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET EXISTENCE CHECK")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    assert TARGET_LOG in df.columns
    assert TARGET_ORIGINAL in df.columns

    print(f"\n✓ {dataset_name}: both target representations available.")

# ----------------------------------------------------------------------
# METADATA CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("METADATA CHECK")
print("=" * 70)

METADATA_COLUMNS = [
    "date",
    "latitude",
    "longitude",
    "data_split"
]

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    missing_metadata = [
        col for col in METADATA_COLUMNS
        if col not in df.columns
    ]

    print(f"\n{dataset_name} missing metadata columns:")
    print(missing_metadata)

    assert len(missing_metadata) == 0

print("\n✓ Required metadata columns available.")

# ----------------------------------------------------------------------
# MISSING VALUE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUE CHECK")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    predictor_missing = df[FEATURES].isna().sum().sum()
    log_target_missing = df[TARGET_LOG].isna().sum()
    original_target_missing = df[TARGET_ORIGINAL].isna().sum()

    print(f"\n{dataset_name}:")
    print("Predictor missing values:", predictor_missing)
    print("Log target missing values:", log_target_missing)
    print("Original target missing values:", original_target_missing)

    assert predictor_missing == 0
    assert log_target_missing == 0
    assert original_target_missing == 0

print("\n✓ No missing predictor or target values.")

# ----------------------------------------------------------------------
# INFINITE VALUE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("INFINITE VALUE CHECK")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    infinite_count = np.isinf(
        df[FEATURES].to_numpy(dtype=np.float64)
    ).sum()

    print(f"\n{dataset_name} infinite predictor values:")
    print(infinite_count)

    assert infinite_count == 0

print("\n✓ No infinite predictor values.")

# ----------------------------------------------------------------------
# TARGET LEAKAGE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET LEAKAGE CHECK")
print("=" * 70)

print("\nLog target included as predictor:")
print(TARGET_LOG in FEATURES)

print("Original CHL included as predictor:")
print(TARGET_ORIGINAL in FEATURES)

assert TARGET_LOG not in FEATURES
assert TARGET_ORIGINAL not in FEATURES

print("\n✓ No target leakage detected.")

# ----------------------------------------------------------------------
# DATE CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("DATE VERIFICATION")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    date_series = pd.to_datetime(df["date"])

    print(f"\n{dataset_name}:")
    print("Minimum date:", date_series.min())
    print("Maximum date:", date_series.max())
    print("Chronologically ordered:",
          date_series.is_monotonic_increasing)

    assert date_series.is_monotonic_increasing

print("\n✓ All datasets are chronologically ordered.")

# ----------------------------------------------------------------------
# DATA SPLIT OVERLAP CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA SPLIT OVERLAP CHECK")
print("=" * 70)

train_dates = set(train_df["date"].astype(str))
validation_dates = set(validation_df["date"].astype(str))
test_dates = set(test_df["date"].astype(str))

train_validation_overlap = len(
    train_dates.intersection(validation_dates)
)

validation_test_overlap = len(
    validation_dates.intersection(test_dates)
)

train_test_overlap = len(
    train_dates.intersection(test_dates)
)

print("\nTrain ∩ Validation:")
print(train_validation_overlap)

print("Validation ∩ Test:")
print(validation_test_overlap)

print("Train ∩ Test:")
print(train_test_overlap)

assert train_validation_overlap == 0
assert validation_test_overlap == 0
assert train_test_overlap == 0

print("\n✓ No date overlap between datasets.")

# ----------------------------------------------------------------------
# SPATIAL CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("SPATIAL INFORMATION CHECK")
print("=" * 70)

for dataset_name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    print(f"\n{dataset_name}:")
    print("Latitude range:",
          df["latitude"].min(),
          "to",
          df["latitude"].max())

    print("Longitude range:",
          df["longitude"].min(),
          "to",
          df["longitude"].max())

print("\n✓ Spatial information preserved.")

# ----------------------------------------------------------------------
# FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 31.9P-14B SUMMARY")
print("=" * 70)

print("\nTraining rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

print("\nSelected predictors:", len(FEATURES))

print("\nLog target:", TARGET_LOG)
print("Original target:", TARGET_ORIGINAL)

print("\n✓ Same 69-feature configuration preserved.")
print("✓ Original and log CHL targets preserved.")
print("✓ No missing predictor values.")
print("✓ No infinite predictor values.")
print("✓ No target leakage.")
print("✓ Chronological ordering preserved.")
print("✓ No date overlap between splits.")
print("✓ Spatial information preserved.")
print("✓ Test data has NOT been used for modelling decisions.")

print("\n" + "=" * 70)
print("STEP 31.9P-14B COMPLETE")
print("=" * 70)

STEP 31.9P-14B — CNN-LSTM DATA LOADING & VERIFICATION
Mounted at /content/drive

Project directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Preprocessing directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing

CNN-LSTM model directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM

Evaluation directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/evaluation

REQUIRED FILE CHECK

Training dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv
Exists: True

Validation dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv
Exists: True

Test dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv
Exists: True

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv
Exis

In [ ]:
# ======================================================================
# STEP 31.9P-14C
# CNN-LSTM TEMPORAL SEQUENCE DESIGN & CONTINUITY ANALYSIS
# ======================================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 31.9P-14C — CNN-LSTM SEQUENCE DESIGN")
print("TEMPORAL SEQUENCE & CONTINUITY ANALYSIS")
print("=" * 70)

# ----------------------------------------------------------------------
# PROJECT PATHS
# ----------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREPROCESS_DIR = (
    BASE_DIR /
    "05_model_preprocessing"
)

MODEL_DIR = (
    BASE_DIR /
    "06_models" /
    "03_CNN_LSTM"
)

EVALUATION_DIR = MODEL_DIR / "evaluation"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ----------------------------------------------------------------------
# FILES
# ----------------------------------------------------------------------

TRAIN_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_train.csv"
)

VALIDATION_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_validation.csv"
)

TEST_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_test.csv"
)

# ----------------------------------------------------------------------
# LOAD DATA
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING DATA")
print("=" * 70)

print("\nLoading training data...")
train_df = pd.read_csv(TRAIN_FILE)

print("Loading validation data...")
validation_df = pd.read_csv(VALIDATION_FILE)

print("Loading test data...")
test_df = pd.read_csv(TEST_FILE)

print("\n✓ All datasets loaded.")

# ----------------------------------------------------------------------
# DATE CONVERSION
# ----------------------------------------------------------------------

for df in [train_df, validation_df, test_df]:
    df["date"] = pd.to_datetime(df["date"])

# ----------------------------------------------------------------------
# BASIC TEMPORAL STRUCTURE
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("1. BASIC TEMPORAL STRUCTURE")
print("=" * 70)

all_dates = pd.concat(
    [
        train_df[["date", "latitude", "longitude"]],
        validation_df[["date", "latitude", "longitude"]],
        test_df[["date", "latitude", "longitude"]]
    ],
    ignore_index=True
)

print("\nTotal rows across all splits:")
print(len(all_dates))

print("\nUnique dates:")
print(all_dates["date"].nunique())

print("\nMinimum date:")
print(all_dates["date"].min())

print("\nMaximum date:")
print(all_dates["date"].max())

# ----------------------------------------------------------------------
# SPATIAL LOCATIONS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("2. SPATIAL LOCATION STRUCTURE")
print("=" * 70)

all_locations = (
    all_dates[
        ["latitude", "longitude"]
    ]
    .drop_duplicates()
    .sort_values(
        ["latitude", "longitude"]
    )
    .reset_index(drop=True)
)

print("\nUnique spatial locations:")
print(len(all_locations))

print("\nFirst 10 spatial locations:")
print(all_locations.head(10).to_string(index=False))

# ----------------------------------------------------------------------
# OBSERVATIONS PER LOCATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("3. OBSERVATIONS PER SPATIAL LOCATION")
print("=" * 70)

location_counts = (
    all_dates
    .groupby(
        ["latitude", "longitude"]
    )
    .size()
    .reset_index(
        name="observations"
    )
)

print("\nObservation count summary:")
print(
    location_counts["observations"]
    .describe()
)

print("\nMinimum observations at one location:")
print(location_counts["observations"].min())

print("Maximum observations at one location:")
print(location_counts["observations"].max())

print("Median observations per location:")
print(location_counts["observations"].median())

# ----------------------------------------------------------------------
# DATE COVERAGE PER LOCATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("4. DATE COVERAGE PER LOCATION")
print("=" * 70)

location_date_summary = (
    all_dates
    .groupby(
        ["latitude", "longitude"]
    )
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        unique_dates=("date", "nunique")
    )
    .reset_index()
)

print(
    "\nDate coverage summary:"
)

print(
    location_date_summary[
        ["unique_dates"]
    ].describe()
)

# ----------------------------------------------------------------------
# DAILY CONTINUITY ANALYSIS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("5. DAILY TEMPORAL CONTINUITY")
print("=" * 70)

continuity_records = []

for (lat, lon), group in all_dates.groupby(
    ["latitude", "longitude"]
):

    dates = (
        group["date"]
        .drop_duplicates()
        .sort_values()
    )

    if len(dates) < 2:
        continue

    gaps = (
        dates
        .diff()
        .dt.days
        .dropna()
    )

    continuity_records.append(
        {
            "latitude": lat,
            "longitude": lon,
            "number_of_dates": len(dates),
            "minimum_gap_days": gaps.min(),
            "maximum_gap_days": gaps.max(),
            "median_gap_days": gaps.median(),
            "mean_gap_days": gaps.mean(),
            "one_day_gap_count": (gaps == 1).sum(),
            "gap_greater_than_1_count": (gaps > 1).sum()
        }
    )

continuity_df = pd.DataFrame(
    continuity_records
)

print("\nContinuity summary:")
print(
    continuity_df[
        [
            "number_of_dates",
            "minimum_gap_days",
            "maximum_gap_days",
            "median_gap_days",
            "mean_gap_days",
            "one_day_gap_count",
            "gap_greater_than_1_count"
        ]
    ].describe()
)

# ----------------------------------------------------------------------
# DAILY GAP PROPORTION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("6. DAILY GAP PROPORTION")
print("=" * 70)

total_consecutive_intervals = (
    continuity_df["one_day_gap_count"]
    +
    continuity_df["gap_greater_than_1_count"]
).sum()

total_one_day_intervals = (
    continuity_df["one_day_gap_count"]
    .sum()
)

if total_consecutive_intervals > 0:

    one_day_percentage = (
        total_one_day_intervals /
        total_consecutive_intervals
    ) * 100

else:

    one_day_percentage = 0

print(
    "\nIntervals exactly 1 day apart:"
)

print(
    total_one_day_intervals
)

print(
    "\nTotal observed intervals:"
)

print(
    total_consecutive_intervals
)

print(
    "\nPercentage of intervals exactly 1 day apart:"
)

print(
    f"{one_day_percentage:.2f}%"
)

# ----------------------------------------------------------------------
# COMMON GAP LENGTHS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("7. MOST COMMON TEMPORAL GAP LENGTHS")
print("=" * 70)

all_gap_values = []

for (lat, lon), group in all_dates.groupby(
    ["latitude", "longitude"]
):

    dates = (
        group["date"]
        .drop_duplicates()
        .sort_values()
    )

    if len(dates) >= 2:

        gaps = (
            dates
            .diff()
            .dt.days
            .dropna()
            .astype(int)
        )

        all_gap_values.extend(
            gaps.tolist()
        )

gap_counts = (
    pd.Series(all_gap_values)
    .value_counts()
    .sort_index()
)

print(
    "\nFirst 20 gap lengths:"
)

print(
    gap_counts.head(20).to_string()
)

# ----------------------------------------------------------------------
# CONTINUOUS RUN ANALYSIS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("8. CONTINUOUS DAILY RUN ANALYSIS")
print("=" * 70)

run_records = []

for (lat, lon), group in all_dates.groupby(
    ["latitude", "longitude"]
):

    dates = (
        group["date"]
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    if len(dates) == 0:
        continue

    run_length = 1

    for i in range(1, len(dates)):

        gap = (
            dates.iloc[i]
            -
            dates.iloc[i - 1]
        ).days

        if gap == 1:

            run_length += 1

        else:

            run_records.append(
                {
                    "latitude": lat,
                    "longitude": lon,
                    "run_length_days": run_length
                }
            )

            run_length = 1

    run_records.append(
        {
            "latitude": lat,
            "longitude": lon,
            "run_length_days": run_length
        }
    )

runs_df = pd.DataFrame(
    run_records
)

print("\nContinuous run summary:")
print(
    runs_df["run_length_days"]
    .describe()
)

# ----------------------------------------------------------------------
# SEQUENCE LENGTH FEASIBILITY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("9. SEQUENCE LENGTH FEASIBILITY")
print("=" * 70)

candidate_sequence_lengths = [
    3,
    5,
    7,
    14,
    21,
    30
]

sequence_feasibility = []

for sequence_length in candidate_sequence_lengths:

    usable_runs = (
        runs_df["run_length_days"]
        >=
        sequence_length
    )

    number_of_runs = (
        usable_runs.sum()
    )

    sequence_feasibility.append(
        {
            "sequence_length_days": sequence_length,
            "continuous_runs_available": number_of_runs,
            "percentage_of_runs_usable": (
                number_of_runs /
                len(runs_df)
            ) * 100
        }
    )

sequence_feasibility_df = pd.DataFrame(
    sequence_feasibility
)

print(
    "\nCandidate sequence lengths:"
)

print(
    sequence_feasibility_df
    .to_string(index=False)
)

# ----------------------------------------------------------------------
# SPLIT BOUNDARIES
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("10. TRAIN / VALIDATION / TEST BOUNDARIES")
print("=" * 70)

split_summary = pd.DataFrame(
    {
        "split": [
            "Training",
            "Validation",
            "Test"
        ],
        "minimum_date": [
            train_df["date"].min(),
            validation_df["date"].min(),
            test_df["date"].min()
        ],
        "maximum_date": [
            train_df["date"].max(),
            validation_df["date"].max(),
            test_df["date"].max()
        ],
        "rows": [
            len(train_df),
            len(validation_df),
            len(test_df)
        ]
    }
)

print(
    split_summary.to_string(
        index=False
    )
)

# ----------------------------------------------------------------------
# CHECK POSSIBLE SEQUENCE CROSSING
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("11. SPLIT BOUNDARY SAFETY")
print("=" * 70)

train_max = train_df["date"].max()
validation_min = validation_df["date"].min()

validation_max = validation_df["date"].max()
test_min = test_df["date"].min()

print("\nTraining last date:")
print(train_max)

print("Validation first date:")
print(validation_min)

print("\nValidation last date:")
print(validation_max)

print("Test first date:")
print(test_min)

train_validation_gap = (
    validation_min - train_max
).days

validation_test_gap = (
    test_min - validation_max
).days

print(
    "\nTrain → Validation gap:"
)

print(
    train_validation_gap,
    "days"
)

print(
    "\nValidation → Test gap:"
)

print(
    validation_test_gap,
    "days"
)

# ----------------------------------------------------------------------
# SAVE ANALYSIS FILES
# ----------------------------------------------------------------------

continuity_file = (
    EVALUATION_DIR /
    "cnn_lstm_temporal_continuity_analysis.csv"
)

sequence_file = (
    EVALUATION_DIR /
    "cnn_lstm_sequence_length_feasibility.csv"
)

location_file = (
    EVALUATION_DIR /
    "cnn_lstm_location_temporal_summary.csv"
)

continuity_df.to_csv(
    continuity_file,
    index=False
)

sequence_feasibility_df.to_csv(
    sequence_file,
    index=False
)

location_date_summary.to_csv(
    location_file,
    index=False
)

# ----------------------------------------------------------------------
# FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 31.9P-14C COMPLETE")
print("=" * 70)

print("\n✓ Spatial locations analysed.")
print("✓ Temporal coverage analysed.")
print("✓ Daily continuity analysed.")
print("✓ Temporal gaps analysed.")
print("✓ Continuous runs analysed.")
print("✓ Candidate sequence lengths evaluated.")
print("✓ Train / validation / test boundaries checked.")
print("✓ No sequences were generated yet.")
print("✓ Test data was NOT used to select a model.")
print("✓ Sequence feasibility files saved.")

print("\nSaved:")
print(continuity_file)
print(sequence_file)
print(location_file)

print("\nIMPORTANT:")
print("Sequence length has NOT been selected automatically.")
print("The results must be reviewed before sequence generation.")

STEP 31.9P-14C — CNN-LSTM SEQUENCE DESIGN
TEMPORAL SEQUENCE & CONTINUITY ANALYSIS

LOADING DATA

Loading training data...
Loading validation data...
Loading test data...

✓ All datasets loaded.

1. BASIC TEMPORAL STRUCTURE

Total rows across all splits:
1052844

Unique dates:
10322

Minimum date:
1997-09-04 00:00:00

Maximum date:
2025-12-31 00:00:00

2. SPATIAL LOCATION STRUCTURE

Unique spatial locations:
102

First 10 spatial locations:
 latitude  longitude
 5.833334  79.333336
 5.833334  79.416664
 5.833334  79.500000
 5.833334  79.583336
 5.833334  79.666664
 5.833334  79.750000
 5.833334  79.833336
 5.833334  79.916664
 5.833334  80.000000
 5.916666  79.333336

3. OBSERVATIONS PER SPATIAL LOCATION

Observation count summary:
count      102.0
mean     10322.0
std          0.0
min      10322.0
25%      10322.0
50%      10322.0
75%      10322.0
max      10322.0
Name: observations, dtype: float64

Minimum observations at one location:
10322
Maximum observations at one location:
10322

In [1]:
# ======================================================================
# STEP 31.9P-14D
# CNN-LSTM SEQUENCE GENERATION
# ======================================================================

import os
import gc
import json
import time
import numpy as np
import pandas as pd

from pathlib import Path
from google.colab import drive

print("=" * 70)
print("STEP 31.9P-14D — CNN-LSTM SEQUENCE GENERATION")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------------------------------------------------

drive.mount("/content/drive", force_remount=False)

# ----------------------------------------------------------------------
# 2. PROJECT DIRECTORIES
# ----------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREPROCESS_DIR = (
    BASE_DIR /
    "05_model_preprocessing"
)

MODEL_DIR = (
    BASE_DIR /
    "06_models" /
    "03_CNN_LSTM"
)

EVALUATION_DIR = (
    MODEL_DIR /
    "evaluation"
)

SEQUENCE_DIR = (
    MODEL_DIR /
    "sequences"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SEQUENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nProject directory:")
print(BASE_DIR)

print("\nCNN-LSTM directory:")
print(MODEL_DIR)

print("\nSequence directory:")
print(SEQUENCE_DIR)

# ----------------------------------------------------------------------
# 3. CONFIGURATION
# ----------------------------------------------------------------------

SEQUENCE_LENGTH = 7
FORECAST_HORIZON = 1

FEATURE_FILE = (
    PREPROCESS_DIR /
    "final_model_feature_columns.csv"
)

TRAIN_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_train.csv"
)

VALIDATION_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_validation.csv"
)

TEST_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_test.csv"
)

LOG_TARGET = "chlorophyll_a_log1p"
ORIGINAL_TARGET = "chlorophyll_a"

METADATA_COLUMNS = [
    "date",
    "latitude",
    "longitude"
]

print("\n" + "=" * 70)
print("SEQUENCE CONFIGURATION")
print("=" * 70)

print("Historical sequence length :", SEQUENCE_LENGTH, "days")
print("Forecast horizon            :", FORECAST_HORIZON, "day")
print("Model target                :", LOG_TARGET)
print("Original target             :", ORIGINAL_TARGET)

# ----------------------------------------------------------------------
# 4. LOAD FINAL FEATURE LIST
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(
    FEATURE_FILE
)

FEATURES = (
    feature_df["feature"]
    .dropna()
    .astype(str)
    .tolist()
)

FEATURES = list(
    dict.fromkeys(FEATURES)
)

print("\nNumber of selected features:")
print(len(FEATURES))

assert len(FEATURES) == 69

print("✓ Exactly 69 predictors confirmed.")

# ----------------------------------------------------------------------
# 5. FUNCTION FOR SEQUENCE COUNT
# ----------------------------------------------------------------------

def count_valid_sequences(
    df,
    sequence_length=7,
    forecast_horizon=1
):

    required_length = (
        sequence_length +
        forecast_horizon
    )

    count = 0

    for (lat, lon), group in df.groupby(
        ["latitude", "longitude"],
        sort=False
    ):

        group = (
            group
            .sort_values("date")
            .reset_index(drop=True)
        )

        dates = (
            group["date"]
            .to_numpy()
            .astype("datetime64[D]")
        )

        n = len(group)

        if n < required_length:
            continue

        # Difference between consecutive dates.
        day_gaps = (
            dates[1:] - dates[:-1]
        ).astype("timedelta64[D]").astype(int)

        # A valid sequence requires
        # sequence_length + forecast_horizon
        # consecutive daily observations.
        for start in range(
            0,
            n - required_length + 1
        ):

            window_gaps = day_gaps[
                start:
                start + required_length - 1
            ]

            if np.all(window_gaps == 1):
                count += 1

    return count

# ----------------------------------------------------------------------
# 6. FUNCTION FOR SEQUENCE GENERATION
# ----------------------------------------------------------------------

def generate_sequences(
    df,
    split_name,
    features,
    sequence_length=7,
    forecast_horizon=1
):

    print("\n" + "=" * 70)
    print(f"GENERATING {split_name.upper()} SEQUENCES")
    print("=" * 70)

    start_time = time.time()

    df = df.copy()

    df["date"] = pd.to_datetime(
        df["date"]
    )

    # --------------------------------------------------------------
    # SAFETY CHECKS
    # --------------------------------------------------------------

    required_columns = (
        features
        +
        [
            LOG_TARGET,
            ORIGINAL_TARGET
        ]
        +
        METADATA_COLUMNS
    )

    missing_columns = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"Missing columns in {split_name}: "
            f"{missing_columns}"
        )

    # --------------------------------------------------------------
    # DUPLICATE LOCATION-DATE CHECK
    # --------------------------------------------------------------

    duplicate_count = (
        df.duplicated(
            subset=[
                "latitude",
                "longitude",
                "date"
            ]
        ).sum()
    )

    print("\nDuplicate location-date rows:")
    print(duplicate_count)

    if duplicate_count != 0:

        raise ValueError(
            f"{split_name} contains duplicate "
            "location-date observations."
        )

    print(
        "✓ One observation per location-date confirmed."
    )

    # --------------------------------------------------------------
    # SORT
    # --------------------------------------------------------------

    df = (
        df
        .sort_values(
            [
                "latitude",
                "longitude",
                "date"
            ]
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------------
    # COUNT VALID SEQUENCES FIRST
    # --------------------------------------------------------------

    print(
        "\nCounting valid continuous sequences..."
    )

    sequence_count = count_valid_sequences(
        df,
        sequence_length,
        forecast_horizon
    )

    print(
        "Valid sequences:",
        sequence_count
    )

    if sequence_count == 0:

        raise ValueError(
            f"No valid sequences found for {split_name}."
        )

    # --------------------------------------------------------------
    # MEMORY ESTIMATION
    # --------------------------------------------------------------

    estimated_values = (
        sequence_count *
        sequence_length *
        len(features)
    )

    estimated_gb = (
        estimated_values *
        np.dtype(np.float32).itemsize
        /
        (1024 ** 3)
    )

    print(
        "\nEstimated X sequence storage:"
    )

    print(
        f"{estimated_gb:.2f} GB"
    )

    # --------------------------------------------------------------
    # CREATE ARRAYS
    # --------------------------------------------------------------

    X = np.empty(
        (
            sequence_count,
            sequence_length,
            len(features)
        ),
        dtype=np.float32
    )

    y_log = np.empty(
        sequence_count,
        dtype=np.float32
    )

    y_original = np.empty(
        sequence_count,
        dtype=np.float32
    )

    target_dates = np.empty(
        sequence_count,
        dtype="datetime64[D]"
    )

    target_latitude = np.empty(
        sequence_count,
        dtype=np.float32
    )

    target_longitude = np.empty(
        sequence_count,
        dtype=np.float32
    )

    # --------------------------------------------------------------
    # GENERATE SEQUENCES
    # --------------------------------------------------------------

    sequence_index = 0

    for location_index, (
        (lat, lon),
        group
    ) in enumerate(
        df.groupby(
            ["latitude", "longitude"],
            sort=False
        )
    ):

        group = (
            group
            .sort_values("date")
            .reset_index(drop=True)
        )

        dates = (
            group["date"]
            .to_numpy()
            .astype("datetime64[D]")
        )

        feature_values = (
            group[features]
            .to_numpy(
                dtype=np.float32
            )
        )

        target_log_values = (
            group[LOG_TARGET]
            .to_numpy(
                dtype=np.float32
            )
        )

        target_original_values = (
            group[ORIGINAL_TARGET]
            .to_numpy(
                dtype=np.float32
            )
        )

        n = len(group)

        if n < (
            sequence_length +
            forecast_horizon
        ):
            continue

        day_gaps = (
            dates[1:] - dates[:-1]
        ).astype("timedelta64[D]").astype(int)

        for start in range(
            0,
            n -
            (
                sequence_length +
                forecast_horizon
            )
            + 1
        ):

            end_input = (
                start +
                sequence_length
            )

            target_index = (
                end_input +
                forecast_horizon -
                1
            )

            required_gaps = day_gaps[
                start:
                target_index
            ]

            # ------------------------------------------------------
            # CONTINUITY CHECK
            # ------------------------------------------------------

            if not np.all(
                required_gaps == 1
            ):
                continue

            # ------------------------------------------------------
            # INPUT SEQUENCE
            # ------------------------------------------------------

            X[sequence_index] = (
                feature_values[
                    start:end_input
                ]
            )

            # ------------------------------------------------------
            # FUTURE TARGET
            # ------------------------------------------------------

            y_log[sequence_index] = (
                target_log_values[
                    target_index
                ]
            )

            y_original[sequence_index] = (
                target_original_values[
                    target_index
                ]
            )

            target_dates[sequence_index] = (
                dates[target_index]
            )

            target_latitude[sequence_index] = (
                lat
            )

            target_longitude[sequence_index] = (
                lon
            )

            sequence_index += 1

        if (
            location_index + 1
        ) % 20 == 0:

            print(
                f"Processed locations: "
                f"{location_index + 1}"
            )

    # --------------------------------------------------------------
    # FINAL ARRAY SIZE CHECK
    # --------------------------------------------------------------

    if sequence_index != sequence_count:

        raise ValueError(
            "Generated sequence count does not "
            "match the pre-counted sequence count."
        )

    print(
        "\n✓ All expected sequences generated."
    )

    # --------------------------------------------------------------
    # SANITY CHECKS
    # --------------------------------------------------------------

    print("\n" + "=" * 70)
    print(f"{split_name.upper()} SEQUENCE SAFETY CHECKS")
    print("=" * 70)

    print("\nX shape:")
    print(X.shape)

    print("\ny_log shape:")
    print(y_log.shape)

    print("\ny_original shape:")
    print(y_original.shape)

    print("\nX dtype:")
    print(X.dtype)

    print("\ny_log dtype:")
    print(y_log.dtype)

    print(
        "\nX NaN values:",
        np.isnan(X).sum()
    )

    print(
        "X infinite values:",
        np.isinf(X).sum()
    )

    print(
        "y_log NaN values:",
        np.isnan(y_log).sum()
    )

    print(
        "y_log infinite values:",
        np.isinf(y_log).sum()
    )

    print(
        "y_original NaN values:",
        np.isnan(y_original).sum()
    )

    print(
        "y_original infinite values:",
        np.isinf(y_original).sum()
    )

    assert not np.isnan(X).any()
    assert not np.isinf(X).any()

    assert not np.isnan(y_log).any()
    assert not np.isinf(y_log).any()

    assert not np.isnan(y_original).any()
    assert not np.isinf(y_original).any()

    print(
        "\n✓ No NaN or infinite values."
    )

    # --------------------------------------------------------------
    # TARGET RANGE
    # --------------------------------------------------------------

    print(
        "\nTarget log1p CHL range:"
    )

    print(
        "Minimum:",
        float(y_log.min())
    )

    print(
        "Maximum:",
        float(y_log.max())
    )

    print(
        "\nTarget original CHL range:"
    )

    print(
        "Minimum:",
        float(y_original.min())
    )

    print(
        "Maximum:",
        float(y_original.max())
    )

    # --------------------------------------------------------------
    # SEQUENCE DATE VERIFICATION
    # --------------------------------------------------------------

    print(
        "\n" + "=" * 70
    )

    print(
        "SEQUENCE TEMPORAL VERIFICATION"
    )

    print(
        "=" * 70
    )

    # Verify the first few sequences directly.
    verification_records = []

    for i in range(
        min(5, len(X))
    ):

        verification_records.append(
            {
                "sequence_index": i,
                "input_start_date": (
                    target_dates[i]
                    -
                    np.timedelta64(
                        sequence_length,
                        "D"
                    )
                ),
                "target_date": target_dates[i],
                "latitude": target_latitude[i],
                "longitude": target_longitude[i],
                "target_log1p": y_log[i],
                "target_original": y_original[i]
            }
        )

    verification_df = pd.DataFrame(
        verification_records
    )

    print(
        verification_df.to_string(
            index=False
        )
    )

    # --------------------------------------------------------------
    # SAVE ARRAYS
    # --------------------------------------------------------------

    split_lower = split_name.lower()

    X_file = (
        SEQUENCE_DIR /
        f"cnn_lstm_X_{split_lower}.npy"
    )

    y_log_file = (
        SEQUENCE_DIR /
        f"cnn_lstm_y_log_{split_lower}.npy"
    )

    y_original_file = (
        SEQUENCE_DIR /
        f"cnn_lstm_y_original_{split_lower}.npy"
    )

    metadata_file = (
        SEQUENCE_DIR /
        f"cnn_lstm_metadata_{split_lower}.csv"
    )

    print(
        "\n" + "=" * 70
    )

    print(
        f"SAVING {split_name.upper()} SEQUENCES"
    )

    print(
        "=" * 70
    )

    np.save(
        X_file,
        X,
        allow_pickle=False
    )

    np.save(
        y_log_file,
        y_log,
        allow_pickle=False
    )

    np.save(
        y_original_file,
        y_original,
        allow_pickle=False
    )

    metadata_df = pd.DataFrame(
        {
            "target_date": pd.to_datetime(
                target_dates
            ),
            "latitude": target_latitude,
            "longitude": target_longitude
        }
    )

    metadata_df.to_csv(
        metadata_file,
        index=False
    )

    print(
        "\nSaved X:"
    )
    print(X_file)

    print(
        "\nSaved log target:"
    )
    print(y_log_file)

    print(
        "\nSaved original target:"
    )
    print(y_original_file)

    print(
        "\nSaved metadata:"
    )
    print(metadata_file)

    # --------------------------------------------------------------
    # FILE SIZES
    # --------------------------------------------------------------

    print(
        "\nSaved file sizes:"
    )

    for file_path in [
        X_file,
        y_log_file,
        y_original_file,
        metadata_file
    ]:

        size_gb = (
            file_path.stat().st_size
            /
            (1024 ** 3)
        )

        print(
            f"{file_path.name}: "
            f"{size_gb:.3f} GB"
        )

    elapsed = (
        time.time() -
        start_time
    )

    print(
        f"\nSequence generation time: "
        f"{elapsed:.2f} seconds"
    )

    # --------------------------------------------------------------
    # RELEASE MEMORY
    # --------------------------------------------------------------

    del X
    del y_log
    del y_original
    del target_dates
    del target_latitude
    del target_longitude
    del df

    gc.collect()

    return {
        "split": split_name,
        "sequence_count": sequence_count,
        "X_file": str(X_file),
        "y_log_file": str(y_log_file),
        "y_original_file": str(y_original_file),
        "metadata_file": str(metadata_file)
    }

# ----------------------------------------------------------------------
# 7. LOAD DATASET
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(
    TRAIN_FILE
)

print(
    "Training shape:",
    train_df.shape
)

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

validation_df = pd.read_csv(
    VALIDATION_FILE
)

print(
    "Validation shape:",
    validation_df.shape
)

print("\n" + "=" * 70)
print("LOADING TEST DATA")
print("=" * 70)

test_df = pd.read_csv(
    TEST_FILE
)

print(
    "Test shape:",
    test_df.shape
)

# ----------------------------------------------------------------------
# 8. GLOBAL INPUT SAFETY CHECK
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("GLOBAL INPUT SAFETY CHECK")
print("=" * 70)

for name, df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    df["date"] = pd.to_datetime(
        df["date"]
    )

    missing_predictors = (
        df[FEATURES]
        .isna()
        .sum()
        .sum()
    )

    missing_log_target = (
        df[LOG_TARGET]
        .isna()
        .sum()
    )

    missing_original_target = (
        df[ORIGINAL_TARGET]
        .isna()
        .sum()
    )

    print(
        f"\n{name}:"
    )

    print(
        "Predictor missing:",
        missing_predictors
    )

    print(
        "Log target missing:",
        missing_log_target
    )

    print(
        "Original target missing:",
        missing_original_target
    )

    assert (
        missing_predictors == 0
    )

    assert (
        missing_log_target == 0
    )

    assert (
        missing_original_target == 0
    )

print(
    "\n✓ All input data passed safety checks."
)

# ----------------------------------------------------------------------
# 9. GENERATE TRAINING SEQUENCES
# ----------------------------------------------------------------------

train_result = generate_sequences(
    train_df,
    "Training",
    FEATURES,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

del train_df
gc.collect()

# ----------------------------------------------------------------------
# 10. GENERATE VALIDATION SEQUENCES
# ----------------------------------------------------------------------

validation_result = generate_sequences(
    validation_df,
    "Validation",
    FEATURES,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

del validation_df
gc.collect()

# ----------------------------------------------------------------------
# 11. GENERATE TEST SEQUENCES
# ----------------------------------------------------------------------

test_result = generate_sequences(
    test_df,
    "Test",
    FEATURES,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

del test_df
gc.collect()

# ----------------------------------------------------------------------
# 12. SAVE SEQUENCE CONFIGURATION
# ----------------------------------------------------------------------

sequence_configuration = {
    "sequence_length_days": SEQUENCE_LENGTH,
    "forecast_horizon_days": FORECAST_HORIZON,
    "number_of_features": len(FEATURES),
    "features": FEATURES,
    "log_target": LOG_TARGET,
    "original_target": ORIGINAL_TARGET,
    "sequence_generation_rule": (
        "7 consecutive historical daily observations "
        "within the same spatial location are used "
        "to predict CHL on the following day."
    ),
    "split_boundary_rule": (
        "Sequences are generated independently within "
        "training, validation and test datasets."
    ),
    "temporal_gap_rule": (
        "Sequences are generated only when all required "
        "observations are exactly one day apart."
    ),
    "test_data_used_for_model_selection": False
}

configuration_file = (
    EVALUATION_DIR /
    "cnn_lstm_sequence_configuration.json"
)

with open(
    configuration_file,
    "w"
) as f:

    json.dump(
        sequence_configuration,
        f,
        indent=4
    )

# ----------------------------------------------------------------------
# 13. FINAL SUMMARY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 31.9P-14D COMPLETE")
print("=" * 70)

print(
    "\nSequence length:"
)

print(
    f"{SEQUENCE_LENGTH} historical days"
)

print(
    "\nForecast horizon:"
)

print(
    f"{FORECAST_HORIZON} day ahead"
)

print(
    "\nFeatures:"
)

print(
    len(FEATURES)
)

print(
    "\nTraining sequences:"
)

print(
    train_result["sequence_count"]
)

print(
    "\nValidation sequences:"
)

print(
    validation_result["sequence_count"]
)

print(
    "\nTest sequences:"
)

print(
    test_result["sequence_count"]
)

print(
    "\n✓ Sequences generated separately by split."
)

print(
    "✓ Spatial locations kept separate."
)

print(
    "✓ Chronological ordering preserved."
)

print(
    "✓ Only consecutive daily observations used."
)

print(
    "✓ Invalid temporal gaps excluded."
)

print(
    "✓ No NaN or infinite sequence values."
)

print(
    "✓ Log CHL targets preserved."
)

print(
    "✓ Original CHL targets preserved."
)

print(
    "✓ Test sequences generated only for final evaluation."
)

print(
    "\nSequence configuration saved:"
)

print(
    configuration_file
)

print(
    "\nSequence files saved under:"
)

print(
    SEQUENCE_DIR
)

STEP 31.9P-14D — CNN-LSTM SEQUENCE GENERATION
Mounted at /content/drive

Project directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data

CNN-LSTM directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM

Sequence directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/sequences

SEQUENCE CONFIGURATION
Historical sequence length : 7 days
Forecast horizon            : 1 day
Model target                : chlorophyll_a_log1p
Original target             : chlorophyll_a

LOADING FINAL FEATURE LIST

Number of selected features:
69
✓ Exactly 69 predictors confirmed.

LOADING TRAINING DATA
Training shape: (736950, 75)

LOADING VALIDATION DATA
Validation shape: (157896, 75)

LOADING TEST DATA
Test shape: (157998, 75)

GLOBAL INPUT SAFETY CHECK

Training:
Predictor missing: 0
Log target missing: 0
Original target missing: 0

Validation:
Predictor missing: 0
Log target missing: 0
Original target missing: 0

Test:
Predictor missing: 0
Log t

In [2]:
# ======================================================================
# STEP 31.9P-14E — CNN-LSTM ARCHITECTURE DESIGN
# ======================================================================

import os
import json
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dropout,
    Dense
)

print("=" * 70)
print("STEP 31.9P-14E — CNN-LSTM ARCHITECTURE DESIGN")
print("=" * 70)

# ----------------------------------------------------------------------
# PROJECT DIRECTORIES
# ----------------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

CNN_LSTM_DIR = os.path.join(
    BASE_DIR,
    "06_models",
    "03_CNN_LSTM"
)

EVALUATION_DIR = os.path.join(
    CNN_LSTM_DIR,
    "evaluation"
)

os.makedirs(EVALUATION_DIR, exist_ok=True)

print("\nProject directory:")
print(BASE_DIR)

print("\nCNN-LSTM directory:")
print(CNN_LSTM_DIR)

print("\nEvaluation directory:")
print(EVALUATION_DIR)

# ----------------------------------------------------------------------
# ARCHITECTURE CONFIGURATION
# ----------------------------------------------------------------------

SEQUENCE_LENGTH = 7
N_FEATURES = 69

print("\n" + "=" * 70)
print("INPUT CONFIGURATION")
print("=" * 70)

print("Sequence length :", SEQUENCE_LENGTH)
print("Number of features :", N_FEATURES)
print("Input shape :", (SEQUENCE_LENGTH, N_FEATURES))

# ----------------------------------------------------------------------
# MODEL ARCHITECTURE
# ----------------------------------------------------------------------

model = Sequential(
    [
        Input(
            shape=(SEQUENCE_LENGTH, N_FEATURES),
            name="input_sequence"
        ),

        Conv1D(
            filters=64,
            kernel_size=3,
            activation="relu",
            padding="same",
            name="cnn_conv1d"
        ),

        MaxPooling1D(
            pool_size=2,
            name="cnn_maxpool"
        ),

        LSTM(
            units=64,
            return_sequences=False,
            name="lstm"
        ),

        Dropout(
            rate=0.20,
            name="dropout"
        ),

        Dense(
            units=32,
            activation="relu",
            name="dense"
        ),

        Dense(
            units=1,
            activation="linear",
            name="output_chl_log1p"
        )
    ],
    name="CNN_LSTM_CHL_Forecaster"
)

# ----------------------------------------------------------------------
# DISPLAY ARCHITECTURE
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("CNN-LSTM ARCHITECTURE")
print("=" * 70)

model.summary()

# ----------------------------------------------------------------------
# PARAMETER COUNT
# ----------------------------------------------------------------------

total_params = model.count_params()

print("\n" + "=" * 70)
print("MODEL PARAMETER COUNT")
print("=" * 70)

print("Total trainable parameters:", total_params)

# ----------------------------------------------------------------------
# LAYER INFORMATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("LAYER CONFIGURATION")
print("=" * 70)

architecture_info = [
    {
        "layer": "Input",
        "configuration": "(7, 69)",
        "purpose": "Seven historical days with 69 selected predictors"
    },
    {
        "layer": "Conv1D",
        "configuration": "64 filters, kernel size 3, ReLU",
        "purpose": "Extract local short-term temporal patterns"
    },
    {
        "layer": "MaxPooling1D",
        "configuration": "pool size 2",
        "purpose": "Reduce temporal representation"
    },
    {
        "layer": "LSTM",
        "configuration": "64 units",
        "purpose": "Learn temporal dependencies"
    },
    {
        "layer": "Dropout",
        "configuration": "0.20",
        "purpose": "Reduce overfitting"
    },
    {
        "layer": "Dense",
        "configuration": "32 units, ReLU",
        "purpose": "Learn nonlinear feature combinations"
    },
    {
        "layer": "Output",
        "configuration": "1 unit, linear",
        "purpose": "Predict next-day CHL log1p"
    }
]

for item in architecture_info:
    print("\nLayer:", item["layer"])
    print("Configuration:", item["configuration"])
    print("Purpose:", item["purpose"])

# ----------------------------------------------------------------------
# OUTPUT CONFIGURATION
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("OUTPUT CONFIGURATION")
print("=" * 70)

print("Model output:", "chlorophyll_a_log1p")
print("Output units:", 1)
print("Activation:", "linear")
print("Forecast:", "1 day ahead")

# ----------------------------------------------------------------------
# SAVE ARCHITECTURE SUMMARY
# ----------------------------------------------------------------------

architecture_file = os.path.join(
    EVALUATION_DIR,
    "cnn_lstm_architecture_summary.json"
)

architecture_summary = {
    "model_name": "CNN_LSTM_CHL_Forecaster",
    "sequence_length_days": SEQUENCE_LENGTH,
    "number_of_features": N_FEATURES,
    "input_shape": [
        SEQUENCE_LENGTH,
        N_FEATURES
    ],
    "forecast_horizon_days": 1,
    "target": "chlorophyll_a_log1p",
    "original_target": "chlorophyll_a",
    "layers": architecture_info,
    "total_parameters": int(total_params),
    "training_started": False
}

with open(architecture_file, "w") as f:
    json.dump(
        architecture_summary,
        f,
        indent=4
    )

print("\n" + "=" * 70)
print("ARCHITECTURE FILE SAVED")
print("=" * 70)

print(architecture_file)

# ----------------------------------------------------------------------
# SAVE MODEL ARCHITECTURE
# ----------------------------------------------------------------------

model_architecture_file = os.path.join(
    EVALUATION_DIR,
    "cnn_lstm_model_architecture.json"
)

with open(model_architecture_file, "w") as f:
    f.write(model.to_json())

print("\nModel architecture JSON:")
print(model_architecture_file)

# ----------------------------------------------------------------------
# FINAL STATUS
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 31.9P-14E COMPLETE")
print("=" * 70)

print("✓ CNN-LSTM architecture created.")
print("✓ Input shape confirmed:", (SEQUENCE_LENGTH, N_FEATURES))
print("✓ Conv1D layer configured.")
print("✓ MaxPooling1D layer configured.")
print("✓ LSTM layer configured.")
print("✓ Dropout layer configured.")
print("✓ Dense layer configured.")
print("✓ Single-output regression layer configured.")
print("✓ Parameter count calculated.")
print("✓ Architecture configuration saved.")
print("✓ Model architecture saved.")

print("\nIMPORTANT:")
print("Training has NOT started.")
print("Validation has NOT been used for training.")
print("Test data has NOT been used.")

STEP 31.9P-14E — CNN-LSTM ARCHITECTURE DESIGN

Project directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data

CNN-LSTM directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM

Evaluation directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/evaluation

INPUT CONFIGURATION
Sequence length : 7
Number of features : 69
Input shape : (7, 69)

CNN-LSTM ARCHITECTURE


Model: "CNN_LSTM_CHL_Forecaster"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ cnn_conv1d (Conv1D)             │ (None, 7, 64)          │        13,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_maxpool (MaxPooling1D)      │ (None, 3, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_chl_log1p (Dense)        │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,449 (189.25 KB)

 Trainable params: 48,449 (189.25 KB)

 Non-trainable params: 0 (0.00 B)


MODEL PARAMETER COUNT
Total trainable parameters: 48449

LAYER CONFIGURATION

Layer: Input
Configuration: (7, 69)
Purpose: Seven historical days with 69 selected predictors

Layer: Conv1D
Configuration: 64 filters, kernel size 3, ReLU
Purpose: Extract local short-term temporal patterns

Layer: MaxPooling1D
Configuration: pool size 2
Purpose: Reduce temporal representation

Layer: LSTM
Configuration: 64 units
Purpose: Learn temporal dependencies

Layer: Dropout
Configuration: 0.20
Purpose: Reduce overfitting

Layer: Dense
Configuration: 32 units, ReLU
Purpose: Learn nonlinear feature combinations

Layer: Output
Configuration: 1 unit, linear
Purpose: Predict next-day CHL log1p

OUTPUT CONFIGURATION
Model output: chlorophyll_a_log1p
Output units: 1
Activation: linear
Forecast: 1 day ahead

ARCHITECTURE FILE SAVED
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/evaluation/cnn_lstm_architecture_summary.json

Model architecture JSON:
/content/drive/MyDrive/Indhi_Mari